<a href="https://colab.research.google.com/github/purimotyann/portfolio/blob/main/task04_35_HT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **〇×ゲーム**




## **概要**
　このプログラムは、Google Colab上で起動して遊ぶことができる、視覚的で扱いやすいボタン操作式（GUI）の〇×ゲームです。

これまでに学習したPythonの基礎知識や応用技術を活用し、1台のPCでご家族やお友達と交互に入力して遊べる「2人対戦（ローカル対戦）モード」と、一人でもじっくり思考を楽しめる「コンピュータ対戦（AI対戦）モード」の2つのプレイモードを搭載いたしました。さらに本プログラムでは、通常の「3×3マス（3個並んだら勝ち）」という標準的なルールに加えて、広いフィールドで頭脳戦を楽しめる「10×10マス（5個繋げたら勝ちの五目並べ方式）」という拡張ルールを設定メニューから自由に選択してプレイすることが可能です。

画面上に生成されるボタン群は、初期状態ではシンプルな空白のマス目として表示されており、マウスで直接クリックするだけで直感的にマークを設置できる仕様となっております。プログラミングの学習成果として、エラーなく動く正確性と、プレイヤーが快適に遊べる操作性の両立を目指して制作いたしました。

## **操作方法**
　プログラムの起動：該当するコードセルを選択した状態で、キーボードの Ctrl + Enterを押すか、左側の実行ボタンをクリックしてプログラムを起動します。

ルールおよびモードの設定：起動後に画面上に現れる設定メニューから、以下の2つの項目を選択します。

【1】対戦モード：「1: 2人対戦」または「2: コンピュータ対戦」のいずれかをラジオボタンで選択します。

【2】ゲームルール：「通常ルール（3x3マス / 3個並んだら勝ち）」または「広大スペース（10x10マス / 5個繋げたら勝ち）」のいずれかをラジオボタンで選択します。

ゲームの開始：設定を選択したら、下部に表示される緑色の「ゲーム開始！」ボタンをクリックします。選択したルールに応じたグリッド状のボタン盤面が自動生成されます。

マーク（〇・✕）の配置：盤面上の置きたい場所の空欄マス（ボタン）をマウスで直接クリックします。自分の手番に応じたマーク（〇または✕）が自動的に色付きで配置され、一度押したマスは誤操作を防ぐため自動的に無効化されます。

勝利判定および終了：選択したルールに従い、縦・横・斜めのいずれかに自分のマークを規定数（3個または5個）先に一直線に並べたプレイヤーの勝利となります。盤面がすべて埋まっても決着がつかない場合は引き分けとなります。

再戦の手順：ゲーム終了後、画面下に表示される黄色の「メニューへ戻る」ボタンをクリックすると設定画面へ戻り、何度でもルールを変更して再プレイが可能です。


## **製作者のコメント**
　課題制作にあたり、最初は授業で習った範囲で完結する単純な「〇×ゲーム」を作ろうと考えておりました。しかし、過去の受講生の制作物を確認したところ、Web AudioでBGMや効果音を鳴らすような完成度の高い作品があり、そのままではオリジナリティも技術力も見劣りしてしまうと感じました。かといって高度なコードをそのままコピー（丸パクリ）するのは自分の学習にならないため、生成AIのアドバイスや過去作の構造を参考にしながら、授業では習っていない画面作成用のライブラリ（ipywidgets）を自分なりに調べて組み込むことに挑戦いたしました。

丸パクリを避け自分自身の作品にするための独自の工夫として、単なる3×3の〇×ゲームで終わらせず、「大きなスペース（10×10マス）の中で5個マークを繋げたら勝利となる五目並べ風の拡張ルール」を自由に変更できるオプション機能として追加いたしました。また、画面の見た目にもこだわり、マス目の中に数字などを出さず、クリックした場所だけに〇や✕が配置されるシンプルでスッキリとしたGUIデザインを採用いたしました。

制作過程で特に苦労した点は、授業で習った def 関数や if 条件分岐だけではカバーできなかった「ボタンイベントの同期処理」や「3×3と10×10の両方に対応できる汎用的な5連・3連勝利判定の多重ループ処理」の構築です。生成AIを活用してコードの壁にぶつかった際の解決策を調べたり、レイアウト崩れを防ぐためにボタンサイズやフォントの調整を何度も試行錯誤いたしました。未履修の技術にも積極的に挑戦したことで、プログラミングへの理解がより深まり、独自の工夫を詰めた自慢の作品を完成させることができました。





## **作成時間**
作成時間：約8時間


In [1]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
from functools import partial

# --- グローバル変数 ---
board = []
board_size = 3
win_length = 3
current_player = "〇"
game_mode = "1"      # "1": 2人対戦, "2": コンピュータ対戦
buttons = []

output_area = widgets.Output()

# --- 判定ロジック（3並び / 5並び対応） ---
def check_win(b, size, target, mark):
    # 1. 横方向
    for r in range(size):
        for c in range(size - target + 1):
            if all(b[r * size + c + i] == mark for i in range(target)):
                return True
    # 2. 縦方向
    for c in range(size):
        for r in range(size - target + 1):
            if all(b[(r + i) * size + c] == mark for i in range(target)):
                return True
    # 3. 斜め（右下方向）
    for r in range(size - target + 1):
        for c in range(size - target + 1):
            if all(b[(r + i) * size + (c + i)] == mark for i in range(target)):
                return True
    # 4. 斜め（左下方向）
    for r in range(size - target + 1):
        for c in range(target - 1, size):
            if all(b[(r + i) * size + (c - i)] == mark for i in range(target)):
                return True
    return False

def check_draw(b):
    return all(cell != "" for cell in b)

# --- コンピュータの思考ロジック ---
def get_computer_move():
    total_cells = board_size * board_size

    # 1. 自分が勝てる手があれば置く
    for i in range(total_cells):
        if board[i] == "":
            board[i] = "✕"
            if check_win(board, board_size, win_length, "✕"):
                board[i] = ""
                return i
            board[i] = ""

    # 2. 相手（〇）が勝ちそうなら防ぐ
    for i in range(total_cells):
        if board[i] == "":
            board[i] = "〇"
            if check_win(board, board_size, win_length, "〇"):
                board[i] = ""
                return i
            board[i] = ""

    # 3. 10x10モード時は既にあるマスの周辺を優先
    if board_size > 3:
        placed = [i for i in range(total_cells) if board[i] != ""]
        candidates = set()
        for p in placed:
            r, c = p // board_size, p % board_size
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < board_size and 0 <= nc < board_size:
                        n_idx = nr * board_size + nc
                        if board[n_idx] == "":
                            candidates.add(n_idx)
        if candidates:
            return random.choice(list(candidates))

    # 4. 中央またはランダム
    center_idx = total_cells // 2
    if board[center_idx] == "":
        return center_idx

    empty_cells = [i for i in range(total_cells) if board[i] == ""]
    return random.choice(empty_cells)

# --- ゲーム進行処理 ---
def on_button_click(idx, b):
    global current_player

    if board[idx] != "":
        return

    make_move(idx, current_player)

    if check_win(board, board_size, win_length, current_player):
        end_game(f"🎉 プレイヤー {current_player} の勝利です！（{win_length}個並びました！）")
        return
    elif check_draw(board):
        end_game("🤝 引き分けです！")
        return

    current_player = "✕" if current_player == "〇" else "〇"
    update_status(f"現在のターン: 【 {current_player} 】")

    if game_mode == "2" and current_player == "✕":
        comp_idx = get_computer_move()
        make_move(comp_idx, "✕")

        if check_win(board, board_size, win_length, "✕"):
            end_game("🤖 コンピュータ（✕）の勝利です！")
            return
        elif check_draw(board):
            end_game("🤝 引き分けです！")
            return

        current_player = "〇"
        update_status(f"現在のターン: 【 {current_player} 】")

def make_move(idx, mark):
    board[idx] = mark
    btn = buttons[idx]
    btn.description = mark
    btn.disabled = True
    if mark == "〇":
        btn.style.button_color = "#ffebee" # うすい赤
    else:
        btn.style.button_color = "#e8eaf6" # うすい青

def end_game(message):
    for btn in buttons:
        btn.disabled = True
    update_status(message)

def update_status(msg):
    with output_area:
        clear_output()
        print(msg)

# --- ゲーム開始処理 ---
def start_game(mode, rule_choice):
    global board, current_player, game_mode, buttons, board_size, win_length

    game_mode = mode
    if rule_choice == "10":
        board_size = 10
        win_length = 5
        btn_wh = "38px"
        font_sz = "12px"
    else:
        board_size = 3
        win_length = 3
        btn_wh = "90px"
        font_sz = "28px"

    total_cells = board_size * board_size
    board = ["" for _ in range(total_cells)]
    current_player = "〇"

    grid_rows = []
    buttons = []
    for i in range(board_size):
        row_btns = []
        for j in range(board_size):
            idx = i * board_size + j
            btn = widgets.Button(
                description="",  # 数字を出さず、空白に設定
                layout=widgets.Layout(width=btn_wh, height=btn_wh, margin="1px")
            )
            btn.style.font_size = font_sz
            btn.style.font_weight = "bold"
            btn.style.button_color = "#f5f5f5"
            btn.on_click(partial(on_button_click, idx))
            row_btns.append(btn)
            buttons.append(btn)
        grid_rows.append(widgets.HBox(row_btns))

    board_widget = widgets.VBox(grid_rows)

    clear_output()
    print(f"=== 〇×ゲーム ({board_size}x{board_size} モード) ===")
    mode_text = "2人対戦モード" if mode == "1" else "コンピュータ対戦モード"
    print(f"モード: {mode_text}（{win_length}個繋げたら勝ち！）\n")
    display(board_widget)
    display(output_area)
    update_status(f"ゲーム開始！ 現在のターン: 【 {current_player} 】")

    restart_btn = widgets.Button(description="メニューへ戻る", button_style="warning")
    restart_btn.on_click(lambda b: show_menu())
    display(restart_btn)

# --- 設定メニュー表示 ---
def show_menu():
    clear_output()
    print("=== 〇×ゲーム 設定メニュー ===")

    # 1. 対戦モード選択
    print("\n【1】 対戦モードを選んでください:")
    mode_select = widgets.RadioButtons(
        options=[('1: 2人対戦', '1'), ('2: コンピュータ対戦', '2')],
        value='1'
    )
    display(mode_select)

    # 2. ルール・盤面選択
    print("\n【2】 ゲームルール（盤面サイズ）を選んでください:")
    rule_select = widgets.RadioButtons(
        options=[
            ('通常ルール（3x3マス / 3個並んだら勝ち）', '3'),
            ('広大スペース（10x10マス / 5個繋げたら勝ち）', '10')
        ],
        value='3'
    )
    display(rule_select)

    start_btn = widgets.Button(
        description="ゲーム開始！",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px", margin="10px 0px")
    )
    start_btn.on_click(lambda b: start_game(mode_select.value, rule_select.value))
    display(start_btn)

# スタート
show_menu()

=== 〇×ゲーム 設定メニュー ===

【1】 対戦モードを選んでください:


RadioButtons(options=(('1: 2人対戦', '1'), ('2: コンピュータ対戦', '2')), value='1')


【2】 ゲームルール（盤面サイズ）を選んでください:


RadioButtons(options=(('通常ルール（3x3マス / 3個並んだら勝ち）', '3'), ('広大スペース（10x10マス / 5個繋げたら勝ち）', '10')), value='3')

Button(button_style='success', description='ゲーム開始！', layout=Layout(height='40px', margin='10px 0px', width='20…